# Fast Iterative Method (FIM) in CUDA

Implementazione passo-passo degli **Algoritmi 6 e 7** della tesi (cap. 3.3), con test finale su una **sorgente puntiforme** in mezzo omogeneo (cap. 4.1).

**Prima di tutto:** *Runtime → Cambia tipo di runtime → Acceleratore hardware: T4 GPU*.

## Controllo GPU
Da eseguire all'inizio di ogni sessione Colab: deve comparire una **Tesla T4**. Se non c'è: *Runtime → Cambia tipo di runtime → T4 GPU*.

In [ ]:
!nvidia-smi

: 

## Step 1 – Algoritmo 2 + Algoritmo 6 in versione sequenziale (CPU)

Obiettivo: capire la logica del FIM **senza** la complessità della GPU e ottenere una **soluzione di riferimento** per gli step CUDA.

Il codice è diviso in due file:
- **`fim_cpu.h`** – "libreria" usata anche dallo Step 2:
  - `solve_quadratic` → **Algoritmo 2** (eq. 3.1), `__host__ __device__`;
  - `solve_node` → la *g(x)* della tesi: legge i 4 vicini (INF fuori dal dominio) e chiama l'Algoritmo 2. Con `rotated = true` aggiunge la **rotazione dello stencil** (§2.6.2): stesso Algoritmo 2 sui vicini diagonali con passo √2·h, e si prende il minimo. Riceve puntatori ed è `__host__ __device__`, quindi la usano sia la CPU sia i kernel GPU;
  - `test_neighbors` → il "test dei nodi adiacenti" dell'Algoritmo 6 (4 vicini, oppure 8 con la rotazione);
  - `fim_cpu` → il ciclo `while A ≠ ∅` dell'**Algoritmo 6**;
  - `relative_error` → eq. (4.2), confronto con U = √(x²+y²).
- **`step1.cu`** – `run_test` (sorgente puntiforme, N = 51, 101, 201, 401) e `main`, che esegue il test due volte: **stencil sugli assi** (FIM della tesi) e **assi + ruotato**.

In [ ]:
%%writefile fim_cpu.h
// fim_cpu.h: funzioni condivise tra CPU e GPU + FIM sequenziale di riferimento.
// Incluso da step1.cu (test CPU) e da step2.cu (confronto GPU vs CPU).
#pragma once
#include <cmath>
#include <vector>

// Valore "infinito": nodo non ancora raggiunto dal fronte d'onda.
// Usiamo un numero finito molto grande (non std::numeric_limits<float>::infinity())
// cosi' che a - b tra due INF faccia 0 e non NaN.
const float INF = 1e30f;

const float SQRT2 = 1.41421356f;

// Spostamenti (di, dj) verso i vicini.
// I primi 4 sono quelli degli assi (sinistra, destra, sotto, sopra);
// gli ultimi 4 sono i diagonali, usati solo con la rotazione dello stencil.
const int DI[8] = {  0, 0, -1, 1,   -1, -1,  1, 1 };
const int DJ[8] = { -1, 1,  0, 0,   -1,  1, -1, 1 };

// ---------------------------------------------------------------------------
// Algoritmo 2 della tesi: soluzione dell'equazione quadratica (3.1)
//   (u - a)^2 + (u - b)^2 = (F h)^2
// a = minimo dei vicini lungo x, b = minimo dei vicini lungo y.
// __host__ __device__: la stessa funzione servira' anche nei kernel GPU.
// ---------------------------------------------------------------------------
__host__ __device__ float solve_quadratic(float a, float b, float F, float h)
{
    float Fh = F * h;
    if (fabsf(a - b) <= Fh)      // il fronte arriva da entrambe le direzioni
        return 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));
    return fminf(a, b) + Fh;     // il fronte arriva da una sola direzione
}

// ---------------------------------------------------------------------------
// g(x) della tesi: calcola il nuovo valore nel nodo (i, j).
// Legge i 4 vicini (INF se fuori dal dominio) e chiama l'Algoritmo 2.
// Con rotated = true applica anche la rotazione dello stencil (tesi, §2.6.2):
// lo stesso Algoritmo 2 sui vicini diagonali con passo sqrt(2) h, e si tiene il minimo.
// Restituisce min(u_vecchio, u_nuovo): u puo' solo diminuire (eq. 3.2).
// Riceve puntatori (non std::vector) ed e' __host__ __device__:
// la CPU la chiama con u.data(), i kernel GPU con i puntatori alla memoria della GPU.
// ---------------------------------------------------------------------------
__host__ __device__ float solve_node(const float* u, const float* F,
                                     int i, int j, int N, float h, bool rotated)
{
    float left  = (j > 0)     ? u[i * N + (j - 1)] : INF;
    float right = (j < N - 1) ? u[i * N + (j + 1)] : INF;
    float down  = (i > 0)     ? u[(i - 1) * N + j] : INF;
    float up    = (i < N - 1) ? u[(i + 1) * N + j] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);

    int idx = i * N + j;
    float q = solve_quadratic(a, b, F[idx], h);           // stencil sugli assi

    if (rotated) {
        // Stencil ruotato di pi/4: i "nuovi assi" sono le due diagonali.
        float down_left  = (i > 0 && j > 0)         ? u[(i - 1) * N + (j - 1)] : INF;
        float up_right   = (i < N - 1 && j < N - 1) ? u[(i + 1) * N + (j + 1)] : INF;
        float up_left    = (i < N - 1 && j > 0)     ? u[(i + 1) * N + (j - 1)] : INF;
        float down_right = (i > 0 && j < N - 1)     ? u[(i - 1) * N + (j + 1)] : INF;

        float a_rot = fminf(down_left, up_right);         // minimo lungo la diagonale "in salita a destra"
        float b_rot = fminf(up_left, down_right);         // minimo lungo la diagonale "in salita a sinistra"
        float q_rot = solve_quadratic(a_rot, b_rot, F[idx], SQRT2 * h);

        q = fminf(q, q_rot);                              // causalita': vince il minore
    }
    return fminf(u[idx], q);
}

// ---------------------------------------------------------------------------
// Test dei nodi adiacenti (parte finale dell'Algoritmo 6).
// Per ogni vicino di x che NON e' in A: se g(x_nb) migliora u(x_nb),
// lo aggiorniamo e lo aggiungiamo ad A.
// Con la rotazione un nodo dipende anche dai diagonali, quindi i vicini sono 8.
// ---------------------------------------------------------------------------
void test_neighbors(int x, std::vector<float>& u, const std::vector<float>& F,
                    int N, float h, int src, bool rotated,
                    std::vector<char>& inA, std::vector<int>& newA)
{
    int i = x / N, j = x % N;
    int n_nb = rotated ? 8 : 4;
    for (int k = 0; k < n_nb; ++k) {
        int ni = i + DI[k], nj = j + DJ[k];
        if (ni < 0 || ni >= N || nj < 0 || nj >= N) continue;  // fuori dal dominio
        int nb = ni * N + nj;
        if (nb == src || inA[nb]) continue;                     // sorgente fissa / gia' in A

        float p = u[nb];
        float q = solve_node(u.data(), F.data(), ni, nj, N, h, rotated);
        if (p > q) {
            u[nb] = q;
            inA[nb] = 1;
            newA.push_back(nb);
        }
    }
}

// ---------------------------------------------------------------------------
// Algoritmo 6 (Fast Iterative Method), versione sequenziale.
// Restituisce il numero di iterazioni del ciclo while.
// ---------------------------------------------------------------------------
int fim_cpu(std::vector<float>& u, const std::vector<float>& F,
            int N, float h, int src, float eps, bool rotated)
{
    std::vector<char> inA(N * N, 0);   // inA[x] = 1 se x appartiene ad A
    std::vector<int>  A;               // lista dei nodi attivi

    // Inizializzazione: la sorgente e' gia' "convergente",
    // quindi facciamo il test dei suoi vicini -> A = 4 vicini della sorgente.
    test_neighbors(src, u, F, N, h, src, rotated, inA, A);

    int iter = 0;
    while (!A.empty()) {
        ++iter;
        std::vector<int> newA;        // nuova lista attivi
        std::vector<int> converged;   // nodi convergenti in questa iterazione

        // Fase 1: aggiornamento di tutti i nodi di A
        for (int x : A) {
            float p = u[x];
            float q = solve_node(u.data(), F.data(), x / N, x % N, N, h, rotated);
            u[x] = q;
            if (fabsf(p - q) < eps) converged.push_back(x);
            else                    newA.push_back(x);    // resta attivo
        }

        // Fase 2: i convergenti escono da A e testano i loro vicini
        for (int x : converged) inA[x] = 0;
        for (int x : converged) test_neighbors(x, u, F, N, h, src, rotated, inA, newA);

        A.swap(newA);
    }
    return iter;
}

// ---------------------------------------------------------------------------
// Errore relativo, eq. (4.2): ||u - U|| / ||U||, con U = sqrt(x^2 + y^2).
// ---------------------------------------------------------------------------
double relative_error(const std::vector<float>& u, int N, float h)
{
    double num = 0.0, den = 0.0;
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h;
            double y = -1.0 + i * h;
            double U = std::sqrt(x * x + y * y);
            double d = u[i * N + j] - U;
            num += d * d;
            den += U * U;
        }
    return std::sqrt(num / den);
}

In [ ]:
%%writefile step1.cu
#include <cstdio>
#include <chrono>
#include <fstream>
#include "fim_cpu.h"     // solve_quadratic, solve_node, fim_cpu, relative_error

// ---------------------------------------------------------------------------
// Test con la sorgente puntiforme per piu' risoluzioni.
// Per N = 101 stampa i valori attorno alla sorgente e salva la soluzione su file.
// ---------------------------------------------------------------------------
void run_test(bool rotated, const char* filename)
{
    const float eps = 1e-6f;
    const int sizes[] = { 51, 101, 201, 401 };
    double prev_err = 0.0;

    printf("\n===== %s =====\n", rotated ? "Stencil assi + ruotato (8 vicini)" : "Stencil assi (4 vicini)");
    printf("   N        h      iterazioni   errore rel.   ordine   tempo [ms]\n");
    for (int N : sizes) {
        float h   = 2.0f / (N - 1);                  // dominio [-1, 1]^2
        int   src = (N / 2) * N + (N / 2);           // nodo centrale -> (0, 0)

        std::vector<float> u(N * N, INF);
        std::vector<float> F(N * N, 1.0f);           // mezzo omogeneo
        u[src] = 0.0f;

        auto t0 = std::chrono::high_resolution_clock::now();
        int iter = fim_cpu(u, F, N, h, src, eps, rotated);
        auto t1 = std::chrono::high_resolution_clock::now();
        double ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

        double err = relative_error(u, N, h);
        // ordine di convergenza stimato: h si dimezza, quindi log2(e_prec / e)
        if (prev_err > 0.0)
            printf("%5d  %.5f  %8d     %.3e     %5.2f   %9.2f\n", N, h, iter, err, std::log2(prev_err / err), ms);
        else
            printf("%5d  %.5f  %8d     %.3e       -     %9.2f\n", N, h, iter, err, ms);
        prev_err = err;

        if (N == 101) {
            int c = N / 2;
            printf("\n  Valori attorno alla sorgente (in unita' di h), N = 101:\n");
            for (int i = c + 1; i >= c - 1; --i) {
                printf("   ");
                for (int j = c - 1; j <= c + 1; ++j) printf("%7.4f ", u[i * N + j] / h);
                printf("\n");
            }
            printf("  atteso: 1 sugli assi, sqrt(2) = 1.4142 in diagonale\n\n");

            std::ofstream out(filename, std::ios::binary);
            out.write(reinterpret_cast<const char*>(u.data()), N * N * sizeof(float));
        }
    }
}

int main()
{
    run_test(false, "step1_assi.bin");     // FIM della tesi (Algoritmi 2 e 6)
    run_test(true,  "step1_rot.bin");      // + rotazione dello stencil (§2.6.2)
    printf("\nSoluzioni N = 101 salvate in step1_assi.bin e step1_rot.bin\n");
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 step1.cu -o step1 && ./step1

### Visualizzazione (N = 101): stencil sugli assi vs stencil ruotato
In alto: curve di livello numeriche (continue) sovrapposte a quelle analitiche (tratteggiate), come in fig. 4.1.
In basso: errore |u − U| in ogni nodo, con la **stessa scala di colori** per poter confrontare i due casi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 101
x = np.linspace(-1, 1, N)
X, Y = np.meshgrid(x, x)
U = np.sqrt(X**2 + Y**2)                                         # soluzione analitica (eq. 4.3)

casi = [('step1_assi.bin', 'Stencil assi (4 vicini)'),
        ('step1_rot.bin',  'Assi + ruotato (8 vicini)')]
sol = [np.fromfile(f, dtype=np.float32).reshape((N, N)) for f, _ in casi]   # riga i = asse y, colonna j = asse x
err_max = max(np.abs(u - U).max() for u in sol)

fig, ax = plt.subplots(2, 2, figsize=(12, 11))
levels = np.arange(0.2, 1.5, 0.2)

for c, (u, (_, titolo)) in enumerate(zip(sol, casi)):
    c1 = ax[0, c].contour(X, Y, u, levels=levels, colors='tab:blue')
    ax[0, c].contour(X, Y, U, levels=levels, colors='k', linestyles='--')
    ax[0, c].clabel(c1, fmt='%.1f', fontsize=8)
    ax[0, c].plot(0, 0, 'ro')
    ax[0, c].set_aspect('equal')
    ax[0, c].set_title(titolo + '\nFIM (blu) vs analitica (tratteggio)')

    im = ax[1, c].imshow(np.abs(u - U), origin='lower', extent=[-1, 1, -1, 1],
                         cmap='viridis', vmin=0, vmax=err_max)
    ax[1, c].set_title(titolo + '\nerrore |u − U|')
    fig.colorbar(im, ax=ax[1, c], shrink=0.8)

for a in ax.flat:
    a.set_xlabel('x [u.a.]'); a.set_ylabel('y [u.a.]')

plt.tight_layout()
plt.show()

## Step 2 – Algoritmo 6 su GPU (un thread per nodo)

Stesso algoritmo dello Step 1, eseguito in parallelo sulla GPU e **confrontato nodo per nodo** con `fim_cpu`.

| Algoritmo 6 | `step2.cu` |
|---|---|
| A = insieme dei nodi attivi | array di flag `active[N*N]` |
| `while A ≠ ∅` | ciclo sulla CPU `while (h_any)` |
| foreach x ∈ A in parallelo: p, q, u(x) = q, convergenza | `update_kernel` (Fase 1) |
| rimuovi x da A + test dei nodi adiacenti | `neighbors_kernel` (Fase 2, versione *gather*) |
| A è vuota? | flag `any_active` copiato sulla CPU |

Punti chiave:
- **Due kernel separati**: la fine di un kernel è l'unico punto in cui *tutta* la GPU è sincronizzata; la Fase 2 deve vedere i nodi convergenti di tutta la griglia.
- **Gather**: in `neighbors_kernel` ogni thread guarda il *proprio* nodo e si chiede "un mio vicino è appena converso?". Così ogni thread scrive solo nella propria cella: nessun conflitto di scrittura.
- **Race condition innocua**: un thread può leggere il valore vecchio o nuovo di un vicino; per la monotonia di u il risultato finale non cambia, ma il numero di iterazioni (soprattutto con la rotazione) può variare tra un'esecuzione e l'altra.

Cosa aspettarsi: max|GPU−CPU| tra 0 e ~1e-6 (la GPU usa FMA, che arrotonda in modo leggermente diverso); speedup < 1 per N piccoli, > 1 per N grandi.

⚠️ Richiede `fim_cpu.h`: esegui prima la cella `%%writefile fim_cpu.h` dello Step 1.

In [ ]:
%%writefile step2.cu
// step2.cu: Algoritmo 6 (FIM) su GPU, un thread per nodo.
// Il risultato viene confrontato nodo per nodo con fim_cpu (Step 1).
#include <cstdio>
#include <cstdlib>
#include <chrono>
#include <cuda_runtime.h>
#include "fim_cpu.h"     // solve_node (__host__ __device__), fim_cpu, relative_error

#define CUDA_CHECK(call)                                                    \
    do {                                                                    \
        cudaError_t err = (call);                                           \
        if (err != cudaSuccess) {                                           \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,           \
                   cudaGetErrorString(err));                                \
            exit(1);                                                        \
        }                                                                   \
    } while (0)

// ===========================================================================
// KERNEL 1 - Fase 1 dell'Algoritmo 6: aggiornamento dei nodi attivi.
//
//   foreach x in A in parallelo:  p = u(x);  q = g(x);  u(x) = q;
//                                 if |p - q| < eps  ->  x e' convergente
//
// Un thread per nodo della griglia. I thread dei nodi NON attivi escono subito.
// Ogni thread scrive solo conv[idx] e u[idx] del PROPRIO nodo.
// ===========================================================================
__global__ void update_kernel(float* u, const float* F,
                              const unsigned char* active, unsigned char* conv,
                              int N, float h, float eps, bool rotated)
{
    int j = blockIdx.x * blockDim.x + threadIdx.x;   // colonna (x)
    int i = blockIdx.y * blockDim.y + threadIdx.y;   // riga    (y)
    if (i >= N || j >= N) return;
    int idx = i * N + j;

    conv[idx] = 0;                 // azzera il flag dell'iterazione precedente
    if (!active[idx]) return;      // x non e' in A: niente da fare

    float p = u[idx];
    float q = solve_node(u, F, i, j, N, h, rotated);
    u[idx] = q;
    if (fabsf(p - q) < eps) conv[idx] = 1;
}

// ===========================================================================
// KERNEL 2 - Fase 2 dell'Algoritmo 6: i convergenti escono da A e
//            i loro vicini fuori da A vengono testati.
//
// Nella tesi e' il nodo convergente x che "spinge" l'informazione verso i vicini.
// Qui facciamo il contrario: ogni thread guarda il PROPRIO nodo e si chiede
// "c'e' un mio vicino che e' appena converso?". Se si', esegue il test su se stesso.
// Il risultato e' lo stesso, ma ogni thread scrive solo nel proprio nodo:
// nessun conflitto tra thread che scrivono nello stesso punto.
//
// Alla fine, se il nodo e' in A, alza il flag any_active (serve all'host per
// sapere se A e' vuota).
// ===========================================================================
__global__ void neighbors_kernel(float* u, const float* F,
                                 unsigned char* active, const unsigned char* conv,
                                 int* any_active,
                                 int N, float h, int src, bool rotated)
{
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    int i = blockIdx.y * blockDim.y + threadIdx.y;
    if (i >= N || j >= N) return;
    int idx = i * N + j;
    if (idx == src) return;        // la sorgente e' fissa e non entra mai in A

    // "Rimuovi x da A": un nodo appena converso non e' piu' attivo.
    bool inA = active[idx] && !conv[idx];

    if (!inA) {
        // Spostamenti verso i vicini (stessi di DI/DJ in fim_cpu.h).
        // Sono dichiarati qui dentro perche' gli array globali della CPU
        // non sono visibili dal codice GPU.
        const int di[8] = {  0, 0, -1, 1,   -1, -1,  1, 1 };
        const int dj[8] = { -1, 1,  0, 0,   -1,  1, -1, 1 };
        int n_nb = rotated ? 8 : 4;

        bool near_converged = false;
        for (int k = 0; k < n_nb; ++k) {
            int ni = i + di[k], nj = j + dj[k];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N) continue;
            if (conv[ni * N + nj]) { near_converged = true; break; }
        }

        // Test del nodo (identico a test_neighbors della CPU)
        if (near_converged) {
            float p = u[idx];
            float q = solve_node(u, F, i, j, N, h, rotated);
            if (p > q) {
                u[idx] = q;
                inA = true;        // "Aggiungi x_nb ad A"
            }
        }
    }

    active[idx] = inA;
    if (inA) *any_active = 1;      // tutti scrivono lo stesso valore: nessun problema
}

// ===========================================================================
// Ciclo dell'Algoritmo 6 sull'host (CPU).
// La CPU lancia i kernel; i dati restano sempre sulla GPU.
// Restituisce il numero di iterazioni; il tempo GPU (ms) va in *ms.
// ===========================================================================
int fim_gpu(std::vector<float>& u, const std::vector<float>& F,
            int N, float h, int src, float eps, bool rotated, float* ms)
{
    const size_t n = (size_t)N * N;

    // ---- 1. Allocazione della memoria sulla GPU ----
    float *d_u, *d_F;
    unsigned char *d_active, *d_conv;
    int *d_any;
    CUDA_CHECK(cudaMalloc(&d_u,      n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_F,      n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_active, n * sizeof(unsigned char)));
    CUDA_CHECK(cudaMalloc(&d_conv,   n * sizeof(unsigned char)));
    CUDA_CHECK(cudaMalloc(&d_any,    sizeof(int)));

    // ---- 2. Stato iniziale ----
    // A vuota; la sorgente e' marcata "convergente" cosi' il primo neighbors_kernel
    // attiva i suoi vicini (come test_neighbors(src) nella versione CPU).
    std::vector<unsigned char> conv0(n, 0);
    conv0[src] = 1;
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), n * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), n * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_conv, conv0.data(), n, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_active, 0, n));

    // ---- 3. Configurazione: blocchi 16x16 che coprono la griglia N x N ----
    dim3 threads(16, 16);
    dim3 blocks((N + threads.x - 1) / threads.x, (N + threads.y - 1) / threads.y);

    // Eventi CUDA per misurare il tempo sulla GPU
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));

    // ---- 4. Inizializzazione di A: vicini della sorgente ----
    int h_any = 0;
    CUDA_CHECK(cudaMemset(d_any, 0, sizeof(int)));
    neighbors_kernel<<<blocks, threads>>>(d_u, d_F, d_active, d_conv, d_any, N, h, src, rotated);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(&h_any, d_any, sizeof(int), cudaMemcpyDeviceToHost));

    // ---- 5. Ciclo principale: while (A non vuota) ----
    int iter = 0;
    while (h_any) {
        ++iter;

        // Fase 1: aggiornamento dei nodi attivi
        update_kernel<<<blocks, threads>>>(d_u, d_F, d_active, d_conv, N, h, eps, rotated);
        CUDA_CHECK(cudaGetLastError());

        // Fase 2: rimozione dei convergenti e test dei vicini.
        // Parte solo quando update_kernel e' finito su TUTTA la griglia
        // (kernel lanciati uno dopo l'altro vengono eseguiti in ordine).
        CUDA_CHECK(cudaMemset(d_any, 0, sizeof(int)));
        neighbors_kernel<<<blocks, threads>>>(d_u, d_F, d_active, d_conv, d_any, N, h, src, rotated);
        CUDA_CHECK(cudaGetLastError());

        // A e' vuota? Copiamo sull'host il flag (questa copia aspetta la fine del kernel).
        CUDA_CHECK(cudaMemcpy(&h_any, d_any, sizeof(int), cudaMemcpyDeviceToHost));
    }

    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));

    // ---- 6. Risultato sull'host e pulizia ----
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, n * sizeof(float), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_u));
    CUDA_CHECK(cudaFree(d_F));
    CUDA_CHECK(cudaFree(d_active));
    CUDA_CHECK(cudaFree(d_conv));
    CUDA_CHECK(cudaFree(d_any));
    return iter;
}

// ===========================================================================
// Confronto GPU vs CPU per piu' risoluzioni.
// ===========================================================================
void run_test(bool rotated)
{
    const float eps = 1e-6f;
    const int sizes[] = { 101, 401, 1001, 2001 };

    printf("\n===== %s =====\n", rotated ? "Stencil assi + ruotato (8 vicini)" : "Stencil assi (4 vicini)");
    printf("   N   iter CPU  iter GPU  max|GPU-CPU|  errore rel. GPU  t CPU [ms]  t GPU [ms]  speedup\n");
    for (int N : sizes) {
        float h   = 2.0f / (N - 1);
        int   src = (N / 2) * N + (N / 2);
        std::vector<float> F(N * N, 1.0f);

        // CPU (riferimento, Step 1)
        std::vector<float> u_cpu(N * N, INF);
        u_cpu[src] = 0.0f;
        auto t0 = std::chrono::high_resolution_clock::now();
        int it_cpu = fim_cpu(u_cpu, F, N, h, src, eps, rotated);
        auto t1 = std::chrono::high_resolution_clock::now();
        double ms_cpu = std::chrono::duration<double, std::milli>(t1 - t0).count();

        // GPU
        std::vector<float> u_gpu(N * N, INF);
        u_gpu[src] = 0.0f;
        float ms_gpu = 0.0f;
        int it_gpu = fim_gpu(u_gpu, F, N, h, src, eps, rotated, &ms_gpu);

        // Differenza massima nodo per nodo
        float max_diff = 0.0f;
        for (int k = 0; k < N * N; ++k) max_diff = fmaxf(max_diff, fabsf(u_gpu[k] - u_cpu[k]));

        printf("%5d  %7d  %8d     %.2e      %.3e     %9.2f  %9.2f  %7.2f\n",
               N, it_cpu, it_gpu, max_diff, relative_error(u_gpu, N, h), ms_cpu, ms_gpu, ms_cpu / ms_gpu);
    }
}

int main()
{
    // La prima chiamata CUDA inizializza la GPU (~100-200 ms): la facciamo qui,
    // cosi' non finisce dentro la misura del tempo del primo test.
    CUDA_CHECK(cudaFree(0));

    run_test(false);   // FIM della tesi
    run_test(true);    // + rotazione dello stencil
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 step2.cu -o step2 && ./step2